# Step 02. Protein modules, unsupervised

weighted gene co-expression network analysis (WGCNA) is told nothing about the patients. Modules are sets of proteins that move together, so
blocks exist by construction rather than by having been selected to separate known groups.

**Why not select proteins first.** Picking proteins by how well they separate known groups collapses
them onto one axis: on this data the 60 limma-selected proteins correlated at median 0.73 and one
component explained 72% of their variance. There were no modules left to find.

In [1]:
source("../src/paths.R")
suppressMessages(library(WGCNA))
options(stringsAsFactors = FALSE); enableWGCNAThreads(4); set.seed(42)
COHORT <- "A"; POWER <- 12
X <- read.csv(coh("R_cohort-%s_log2_combat.csv", COHORT), row.names = 1, check.names = FALSE)
m <- read.csv(coh("R_cohort-%s_meta.csv",        COHORT), row.names = 1, check.names = FALSE)
m <- m[rownames(X), , drop = FALSE]
dim(X)

Allowing parallel execution with up to 4 working processes.


[1]   87 7288

## `minModuleSize` decides whether the interferon signature exists

The curated interferon set in `proteins/interferon-response-genes.json` has 23 reagents on this
SomaScan menu. Whether they land in a module at all turns entirely on the minimum module size:

| minModuleSize | modules | unassigned (grey) | ISGs in grey | largest interferon-stimulated gene (ISG) module |
|---|---|---|---|---|
| 30 | 11 | 1606 | 11 | blue: 5 of 23 |
| 20 | 21 | 1392 | 10 | blue: 5 of 23 |
| 15 | 30 | 1271 | 11 | blue: 4 of 23 |
| 10 | 38 | 1166 | 6 | skyblue3: 8 of 23 |
| 5 | 52 | 1016 | 3 | ivory: 10 of 23 |

At 30, WGCNA's usual default for expression data, 11 of the 23 ISG reagents fall into grey, and
the best any module does is `blue` with 5, and `blue` is the 1,213-protein module that spans a
large fraction of the panel, so that is not a signature, it is a bucket. The signature looked
absent. It was not absent. It is a 14-protein module, and a floor of 30 cannot represent a
14-protein module. That was a parameter choice, not a property of the data.

The cost is 52 modules instead of 11. Run the scan yourself rather than taking the table on trust, the cell
below is the function that produced it, left commented out because it is five full fits and about
seven minutes. It writes `minmodulesize_scan_A.rds`.

**Lower is not better.** Continuing the scan below 5 buys nothing: at minModuleSize 4, 3 and 2 the
curated ISG capture stalls at 10, 9, 9 while the module count runs 60 → 68 → 96 and the median
module halves from 25 to 13. The interferon module itself holds at 13–14 proteins throughout. Five
is where the signature resolves and the panel has not yet shattered.

In [2]:
# The scan that produced the table above. Five full blockwiseModules fits, about
# 80 seconds each, so it is left commented out -- uncomment to reproduce rather
# than taking the table on trust.
scan_minmod <- function(sizes = c(30, 20, 15, 10, 5)) {
  isg <- read_ifn_panel(colnames(X))
  res <- do.call(rbind, lapply(sizes, function(s) {
    set.seed(42)
    net  <- blockwiseModules(X, power = POWER, networkType = "signed", minModuleSize = s,
                             mergeCutHeight = 0.25, numericLabels = TRUE,
                             maxBlockSize = 8000, verbose = 0)
    mm   <- labels2colors(net$colors)
    hit  <- mm[match(isg, colnames(X))]
    named<- hit[hit != "grey"]
    top  <- if (length(named)) sort(table(named), decreasing = TRUE)[1] else c(none = 0)
    data.frame(minModuleSize = s, modules = length(unique(mm)) - 1, grey = sum(mm == "grey"),
               isg_in_grey = sum(hit == "grey"),
               largest_isg_module = sprintf("%s: %d of %d", names(top), top[[1]], length(isg)))
  }))
  saveRDS(res, art("minmodulesize_scan_%s.rds", COHORT))
  res
}
# scan_minmod()      # ~7 minutes, five fits

In [3]:
net  <- blockwiseModules(X, power = POWER, networkType = "signed", minModuleSize = 5,
                         mergeCutHeight = 0.25, numericLabels = TRUE,
                         maxBlockSize = 8000, verbose = 0)
mods <- labels2colors(net$colors)
ME   <- moduleEigengenes(X, mods)$eigengenes
sort(table(mods), decreasing = TRUE)

mods
      turquoise            blue            grey           brown          yellow 
           2216            1213            1016             504             373 
          green             red           black            pink         magenta 
            204             178             150             138             121 
         purple     greenyellow             tan          salmon            cyan 
            101              78              77              60              44 
   midnightblue       lightcyan          grey60      lightgreen     lightyellow 
             42              40              39              39              38 
      royalblue         darkred       darkgreen   darkturquoise        darkgrey 
             36              34              32              31              30 
     darkorange          orange         skyblue           white     saddlebrown 
             27              27              24              24              23 
      steelblue   palet

## Does this agree with the published analysis of the same data?

The source paper reports 21 modules on n = 207 systemic lupus erythematosus (SLE), with the interferon module being red, 35
proteins, hub ISG15 (SOMAmer seq.14148.2). We are on n = 97, so we expect smaller and more
numerous modules. The test is whether the interferon module reproduces.

In [4]:
# Locate the module by OVERLAP with a curated interferon-stimulated gene set,
# proteins/interferon-response-genes.json -- 37 genes chosen from the interferon
# literature, 19 of them on this SomaScan menu, carrying 23 probes.
#
# NEVER by colour: WGCNA assigns colours by module rank within a single fit, so
# "royalblue" in one run and "royalblue" in another are unrelated.
#
# And no longer by a single hardcoded SOMAmer either. grep("14148") could not
# fail -- it returned whichever module that one probe happened to land in, and
# called it interferon. Overlap with a set CAN fail, and locate_ifn_module()
# stops if no module holds at least three of the set.
ifn <- locate_ifn_module(mods, colnames(X))
cat("curated ISG probes by module:\n"); print(attr(ifn, "hits"))

rb <- colnames(X)[mods == ifn]
cat(sprintf("\ninterferon module = '%s', %d proteins\n\n", ifn, length(rb)))
print(rb)
k <- cor(X[, rb], ME[[paste0("ME", ifn)]])[, 1]
cat("\nour hub by module membership (kME):", rb[which.max(k)],
    sprintf("%.3f", max(k)), "\n")

curated ISG probes by module:


hits
    ivory     brown     green    yellow   magenta      pink       tan turquoise 
       10         2         2         2         1         1         1         1 



interferon module = 'ivory', 14 proteins



 [1] "STAT1_seq.10370.21" "STAT1_seq.12351.25" "DDX58"             
 [4] "IFIT3"              "C1QC"               "ISG15_seq.14148.2" 
 [7] "ISG15_seq.14151.4"  "GBP1"               "LAP3_seq.15610.72" 
[10] "MX1"                "IFIH1"              "ANKRD45"           
[13] "INHA"               "CXCL10"            



our hub by module membership (kME): ISG15_seq.14151.4 0.940 


**The published interferon module is recovered.** `ivory` holds 14 proteins, 10 of them canonical
interferon-stimulated: STAT1 (two SOMAmers), DDX58, IFIT3, ISG15 (both SOMAmers), GBP1, MX1, IFIH1,
CXCL10. Its hub by module membership is ISG15 `seq.14151.4` at kME 0.94, ISG15 is the hub
protein the source paper names for its own interferon module, found there on n = 207 by a different
route.

But the module's boundary moves with the sample, and that has to be said alongside. Earlier
draws of ~90 patients from the same cohort put these proteins in modules of 12, 23 and 39 members,
and one draw dissolved them into a 775-protein module. The module count at identical parameters has
ranged from 25 to 52. The proteins co-cluster reliably; *where the boundary falls* does not.

**WGCNA colours mean nothing across fits.** They are assigned by module rank within one run. This
module has been "royalblue", "darkorange", "blue" and now "ivory" across runs of the same pipeline, always the same biology, never the same name. Nothing here ever names a colour; the module is
looked up every time.

**Looked up by a gene set, not by one probe.** Earlier versions grepped for the SOMAmer `14148` and
took whichever module it landed in. That lookup always returns a module, it always returns a module, and calls
it interferon. `proteins/interferon-response-genes.json` holds 37 curated interferon-stimulated
genes, 19 of them on this menu carrying 23 probes, and `locate_ifn_module()` takes the module
holding a plurality of them and stops if no module holds at least three. On this fit, `ivory`
takes 10; the runner-up takes 2. The set was written from the interferon literature rather than
from this fit's membership, so recovering `ivory` is a result and not a definition.

`modulePreservation` across cohorts B and C is what turns "recovered" into "reproducible", and it
has not been run.

In [5]:
saveRDS(list(cohort = COHORT, X = X, meta = m, power = POWER, mods = mods, ME = ME,
             ifn = ifn, minModuleSize = 5, mergeCutHeight = 0.25, seed = 42),
        art("wgcna_%s.rds", COHORT))
write.csv(data.frame(protein = colnames(X), module = mods),
          art("modules_%s.csv", COHORT), row.names = FALSE)

The fit is saved to `data/run_artifacts/`, which is gitignored, this file is large and entirely
regenerable from `cohorts/` plus these parameters. Every later step reads this artifact and none
of them refits, that is what
makes the steps separable, cacheable, and safe to containerize one per stage.

## Where this was run

Rendered notebooks are committed, so each records the machine, the R and the
package versions that produced its output.


In [6]:
run_provenance()

run on   : Annes-MacBook-Pro-193.local ( Darwin 27.0.0 )
date     : 2026-09-25 12:03 EDT 
R        : R version 4.4.3 (2025-02-28) | x86_64-apple-darwin13.4.0 
R comes from: /Users/adeslatt/miniforge3/envs/endotypes-proteomics 
packages :
   WGCNA            1.74
   ComplexHeatmap   2.22.0
   sva              3.54.0
   VarSelLCM        2.1.3.2
   cluster          2.1.8.1
   fpc              2.2.15
   circlize         0.4.18
